# Band Gap and In-Gap States at a 5|7 Grain Boundary in h-BN

> **Qiucheng Li, Xiaolong Zou, Mengxi Liu, Jingyu Sun, Yabo Gao, Yue Qi, Xiebo Zhou, Boris I. Yakobson, Yanfeng Zhang, and Zhongfan Liu**
> "Grain Boundary Structures and Electronic Properties of Hexagonal Boron Nitride on Cu(111)"
> Nano Letters 15 (9), 5804-5810 (2015)
> [DOI: 10.1021/acs.nanolett.5b01852](https://doi.org/10.1021/acs.nanolett.5b01852)

Computes the local density of states (LDOS) at the Type III 5|7 grain boundary created in the
[structure notebook](defect_planar_grain_boundary_2d_boron_nitride_5_7.ipynb) and of a perfect h-BN layer, and compares
them with Li et al.'s Fig. 4b: the gap between the two LDOS peaks that border it, about 3.2 eV, the four boundary
peaks, and the gap of the perfect layer. Density of States jobs on both cells supply the per-atom projected densities
of states; the LDOS of the boundary is their sum over the atoms next to the two boundaries of the cell.

<h2 style="color:green">Usage</h2>

1. Create the material in the [structure notebook](defect_planar_grain_boundary_2d_boron_nitride_5_7.ipynb), which saves `h-BN 5-7 grain boundary, MACE-relaxed` to the `uploads` folder.
1. Set the material names and parameters in cells 1.2-1.5 (or use the defaults); `RELAX = True` relaxes the boundary with DFT first, once, and saves it as `<name> relaxed` for reuse.
1. Click "Run" > "Run All" to run all cells.
1. Wait for the jobs to complete.
1. Scroll down to view the results.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the material names, model, compute and read-out parameters.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Load the boundary by name and the perfect layer from Standata, print provenance, and save both to the platform.
1. Configure the shared DFT model and k-grid: one model and one k-point density for both cells.
1. Configure compute: get the list of clusters and create a compute configuration.
1. Relax the boundary if `RELAX`, reusing a relaxed structure when one already exists.
1. Create, submit and monitor the Density of States jobs.
1. Retrieve the results: the LDOS of the boundary and of the bulk of the boundary cell, the energy zero, the gap and the peaks, the perfect layer's gap, and the plot.
1. Compare with Li et al. (2015).

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples|api_examples")

### 1.2. Material names

In [ ]:
GRAIN_BOUNDARY_NAME = "h-BN 5-7 grain boundary, MACE-relaxed"  # saved by the structure notebook
# GRAIN_BOUNDARY_NAME = "h-BN 5-7 grain boundary"  # the same boundary as built, before MACE
PERFECT_LAYER_NAME = "BN, Hexagonal Boron Nitride, HEX (P6/mmm) 2D (Monolayer), 2dm-4991"  # Standata

### 1.3. Parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"

DOS_SEARCH_TERM = "dos.json"
RELAX_WORKFLOW_SEARCH_TERM = "fixed_cell_relaxation.json"
APPLICATION_NAME = "espresso"

# False: the MACE-relaxed boundary as saved. True: relax it with DFT to the manuscript's 0.01 eV/Å first, once.
RELAX = False

CLUSTER_NAME = "001"  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.OR
PPN = 16  # OR allows 16 cores per job
TIME_LIMIT = "04:00:00"

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

### 1.4. DFT model parameters

In [ ]:
FUNCTIONAL = "pbe"  # the manuscript's functional (VASP, PAW)
PSEUDOPOTENTIAL_TYPE = "us"  # GBRV ultrasoft, the platform's default PBE family for B and N
ECUTWFC = 40  # Ry, GBRV's recommended wavefunction cutoff
ECUTRHO = 200  # Ry, GBRV's recommended charge-density cutoff

KPOINT_DENSITY = 4  # Å⁻¹, one density for both cells; the manuscript does not state its k-points
MODEL_TAG = f"{FUNCTIONAL}-{PSEUDOPOTENTIAL_TYPE} {ECUTWFC}-{ECUTRHO}Ry k{KPOINT_DENSITY}"

SCF_UNIT = "pw_scf"
NSCF_UNIT = "pw_nscf"
RELAX_UNIT = "pw_relax"
RELAXATION_SETTINGS = {"forc_conv_thr": 3.9e-4, "nstep": 100}  # 0.01 eV/Å, the manuscript's convergence target
# Names the relaxation job; the relaxed structure itself is found by content hash.
RELAX_TAG = f"{MODEL_TAG} f{RELAXATION_SETTINGS['forc_conv_thr']}"


### 1.5. Read-out parameters
The boundaries of the cell lie at x = 0 and at the middle of the cell, as the structure notebook builds them.

In [ ]:
BOUNDARY_DISTANCE = 2.0  # Å, atoms this close to a boundary make up its LDOS
BULK_DISTANCE = 5.0  # Å, atoms farther than this from both boundaries are the bulk, whose mid-gap is the energy zero
GAP_DOS_THRESHOLD = 0.02  # fraction of the maximum DOS below which an energy counts as inside the gap
PEAK_PROMINENCE = 0.1  # fraction of the maximum boundary LDOS in the window a peak must stand out by
ENERGY_WINDOW = (-4.0, 4.0)  # eV, the range of Fig. 4b

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Load the materials
### 3.1. Load the boundary from the uploads folder or the platform, the perfect layer from Standata, and print provenance

In [ ]:
from collections import Counter
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.core.entity.material.api import load_material
from mat3ra.standata.materials import Materials

materials = {
    GRAIN_BOUNDARY_NAME: load_material(client, FOLDER, GRAIN_BOUNDARY_NAME, ACCOUNT_ID),
    PERFECT_LAYER_NAME: Material.create(Materials.get_by_name_first_match(PERFECT_LAYER_NAME)),
}
for name, material in materials.items():
    print(f"{name}: {dict(Counter(material.basis.elements.values))}, {material.basis.number_of_atoms} atoms, "
          f"cell {material.lattice.a:.2f} x {material.lattice.b:.2f} x {material.lattice.c:.2f} Å")

### 3.2. Save the materials to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_materials = {}
for name, material in materials.items():
    material.basis.set_labels_from_list([])
    saved_materials[name] = get_or_create_material(client, material, ACCOUNT_ID)

## 4. Configure the shared DFT model and k-grid
### 4.1. DFT model

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.ade.application import Application
from mat3ra.mode import ModelFactory

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype="gga", functional=FUNCTIONAL)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)
print(f"Using application: {app.name}, model: {MODEL_TAG}")

### 4.2. k-grid per material

In [ ]:
from mat3ra.notebooks_utils.workflow import kgrid_from_density

kgrid = {
    name: kgrid_from_density(material, KPOINT_DENSITY, periodic_dims=(0, 1)) for name, material in materials.items()
}
for name, grid in kgrid.items():
    print(f"{name}: k-grid {grid}")

## 5. Create the compute configuration
### 5.1. Select cluster

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create the compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
    if cluster is None:
        raise ValueError(f"Cluster '{CLUSTER_NAME}' not found. Available: {[c['hostname'] for c in clusters]}")
else:
    cluster = clusters[0]
compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}, "
      f"time limit: {TIME_LIMIT}")

## 6. Relax the grain boundary (optional)

Runs only if `RELAX`: finds any relaxed version of this structure already on the account,
regardless of who relaxed it or with what -- before spending any compute on a new one.

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.workflow import apply_planewave_cutoffs, apply_scf_kgrid, patch_workflow_qe_input
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async
from mat3ra.notebooks_utils.core.entity.job.api import find_job_for_material
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job
from mat3ra.notebooks_utils.core.entity.material.api import find_relaxed_material, get_final_structure_for_job

grain_boundary = materials[GRAIN_BOUNDARY_NAME]
relaxed_grain_boundary = None
if RELAX:
    relax_workflow_name = f"Fixed-cell Relaxation {GRAIN_BOUNDARY_NAME} {RELAX_TAG}"
    relaxed_grain_boundary = find_relaxed_material(client, grain_boundary, ACCOUNT_ID)
    if relaxed_grain_boundary is not None:
        print(f"♻️  Relaxed grain boundary: {relaxed_grain_boundary.name} ({relaxed_grain_boundary.id})")
    else:
        relax_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
            RELAX_WORKFLOW_SEARCH_TERM
        )
        relax_workflow = Workflow.create(relax_workflow_config)
        relax_workflow.name = relax_workflow_name
        relax_workflow.subworkflows[0].model = model
        apply_planewave_cutoffs(relax_workflow, ECUTWFC, ECUTRHO, unit_name=RELAX_UNIT)
        apply_scf_kgrid(relax_workflow, kgrid[GRAIN_BOUNDARY_NAME], material=grain_boundary, unit_name=RELAX_UNIT)
        patch_workflow_qe_input(relax_workflow, {"control": RELAXATION_SETTINGS}, [RELAX_UNIT])

        relax_job = find_job_for_material(
            client, saved_materials[GRAIN_BOUNDARY_NAME]["_id"], relax_workflow_name, ACCOUNT_ID,
            statuses=("submitted", "queued", "active", "finished"),
        )
        if relax_job is None:
            relax_job = create_job(
                api_client=client, materials=[saved_materials[GRAIN_BOUNDARY_NAME]], workflow=relax_workflow,
                project_id=project_id, owner_id=ACCOUNT_ID, compute=compute.to_dict(),
                prefix=f"{relax_workflow_name} {timestamp}",
            )
            submit_jobs(client.jobs, [relax_job["_id"]])
        await wait_for_jobs_to_finish_async(client.jobs, [relax_job["_id"]], poll_interval=POLL_INTERVAL)

        relaxed_material = get_final_structure_for_job(client, relax_job["_id"])
        client.materials.update(relaxed_material.id, {"name": f"{GRAIN_BOUNDARY_NAME} relaxed"})
        relaxed_grain_boundary = Material.create(client.materials.get(relaxed_material.id))
        print(f"✅ Relaxed grain boundary: {relaxed_grain_boundary.name} ({relaxed_grain_boundary.id})")

        total_force = get_properties_for_job(client, relax_job["_id"], "total_force")[0]
        print(f"Residual force after relaxation (norm over all atoms): "
              f"{total_force['value']:.4f} {total_force['units']}")

## 7. Density of States jobs
### 7.1. Create the jobs
One job per cell; a job that already ran under the same workflow name is re-used.

In [ ]:
dos_structure = relaxed_grain_boundary if RELAX else grain_boundary
dos_inputs = {
    GRAIN_BOUNDARY_NAME: relaxed_grain_boundary.to_dict() if RELAX else saved_materials[GRAIN_BOUNDARY_NAME],
    PERFECT_LAYER_NAME: saved_materials[PERFECT_LAYER_NAME],
}
dos_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(DOS_SEARCH_TERM)
dos_job_ids = {}
new_job_ids = []
for name, saved_material in dos_inputs.items():
    dos_workflow = Workflow.create(dos_workflow_config)
    dos_workflow.name = f"DOS {name} {MODEL_TAG}"
    dos_workflow.subworkflows[0].model = model
    for unit_name in (SCF_UNIT, NSCF_UNIT):
        apply_planewave_cutoffs(dos_workflow, ECUTWFC, ECUTRHO, unit_name=unit_name)
        apply_scf_kgrid(dos_workflow, kgrid[name], material=materials[name], unit_name=unit_name)
    job = find_job_for_material(
        client, saved_material["_id"], dos_workflow.name, ACCOUNT_ID,
        statuses=("submitted", "queued", "active", "finished"),
    )
    if job is None:
        job = create_job(
            api_client=client, materials=[saved_material], workflow=dos_workflow, project_id=project_id,
            owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{dos_workflow.name} {timestamp}",
        )
        new_job_ids.append(job["_id"])
        print(f"✅ {name}: created DOS job {job['_id']}")
    else:
        print(f"♻️  {name}: reusing DOS job {job['_id']}")
    dos_job_ids[name] = job["_id"]

### 7.2. Submit and monitor the jobs

In [ ]:
if new_job_ids:
    submit_jobs(client.jobs, new_job_ids)
    print(f"✅ Submitted {len(new_job_ids)} job(s).")
await wait_for_jobs_to_finish_async(client.jobs, list(dos_job_ids.values()), poll_interval=POLL_INTERVAL)

## 8. Retrieve the results
### 8.1. LDOS of the boundaries and of the bulk

Projected densities of states are written per atom and orbital; the LDOS of a set of atoms is their sum per atom.

In [ ]:
import re
from io import StringIO

import numpy as np
from mat3ra.notebooks_utils.io import read_from_url

pdos_urls = {}
for file in client.jobs.list_files(dos_job_ids[GRAIN_BOUNDARY_NAME]):
    for atom_number in re.findall(r"pdos_atm#(\d+)\(", file["key"]):
        pdos_urls.setdefault(int(atom_number) - 1, []).append(file["signedUrl"])

cartesian_structure = dos_structure.clone()
cartesian_structure.to_cartesian()
x_coordinates = np.array(cartesian_structure.basis.coordinates.values)[:, 0]
width = cartesian_structure.lattice.a
boundary_distances = np.min(
    [np.abs((x_coordinates - line + width / 2) % width - width / 2) for line in (0.0, width / 2)], axis=0
)
boundary_atoms = np.where(boundary_distances < BOUNDARY_DISTANCE)[0]
bulk_atoms = np.where(boundary_distances > BULK_DISTANCE)[0]


async def get_ldos(atoms):
    urls = [url for atom in atoms for url in pdos_urls[atom]]
    tables = [np.loadtxt(StringIO(await read_from_url(url)), usecols=(0, 1)) for url in urls]
    return tables[0][:, 0], sum(table[:, 1] for table in tables) / len(atoms)


energies, boundary_ldos = await get_ldos(boundary_atoms)
_, bulk_ldos = await get_ldos(bulk_atoms)
print(f"Boundary: {len(boundary_atoms)} atoms within {BOUNDARY_DISTANCE} Å; "
      f"bulk: {len(bulk_atoms)} atoms beyond {BULK_DISTANCE} Å")

### 8.2. Energy zero, gap and in-gap peaks

The zero is the middle of the gap of the bulk LDOS, as in Fig. 4b. Peaks (2) and (3) are the boundary peaks nearest
below and above it, (1) and (4) the next ones out; the gap is the distance from (2) to (3).

In [ ]:
from scipy.signal import find_peaks


def get_gap_edges(energies, dos, fermi_energy):
    states = energies[dos > GAP_DOS_THRESHOLD * dos.max()]
    return states[states < fermi_energy].max(), states[states > fermi_energy].min()


fermi_energy = get_properties_for_job(client, dos_job_ids[GRAIN_BOUNDARY_NAME], "fermi_energy")[-1]["value"]
bulk_edges = get_gap_edges(energies, bulk_ldos, fermi_energy)
shifted_energies = energies - np.mean(bulk_edges)
window = (shifted_energies >= ENERGY_WINDOW[0]) & (shifted_energies <= ENERGY_WINDOW[1])
peak_indices, _ = find_peaks(boundary_ldos[window], prominence=PEAK_PROMINENCE * boundary_ldos[window].max())
peak_energies = shifted_energies[window][peak_indices]
peaks = [*peak_energies[peak_energies < 0][-2:], *peak_energies[peak_energies > 0][:2]]
gap = peaks[2] - peaks[1]
print(f"Bulk gap: {bulk_edges[1] - bulk_edges[0]:.2f} eV")
print(f"Boundary peaks: {', '.join(f'{peak:+.2f}' for peak in peaks)} eV; gap (2) to (3): {gap:.2f} eV")

### 8.3. Perfect layer

In [ ]:
perfect_files = {
    file["key"].split("/")[-1]: file["signedUrl"] for file in client.jobs.list_files(dos_job_ids[PERFECT_LAYER_NAME])
}
perfect_table = np.loadtxt(StringIO(await read_from_url(perfect_files["__prefix__.pdos_tot"])), usecols=(0, 1))
perfect_dos = perfect_table[:, 1] / materials[PERFECT_LAYER_NAME].basis.number_of_atoms
perfect_fermi_energy = get_properties_for_job(client, dos_job_ids[PERFECT_LAYER_NAME], "fermi_energy")[-1]["value"]
perfect_edges = get_gap_edges(perfect_table[:, 0], perfect_dos, perfect_fermi_energy)
perfect_energies = perfect_table[:, 0] - np.mean(perfect_edges)
perfect_gap = perfect_edges[1] - perfect_edges[0]
print(f"Perfect layer gap: {perfect_gap:.2f} eV")

### 8.4. LDOS of the boundary and of the perfect layer (Fig. 4b of the manuscript)

The manuscript's Fig. 4b (Type I is not computed here):

<img src="https://github.com/Exabyte-io/documentation/raw/af35aa141fe02b1a182289ac779437f60f47ef97/images/tutorials/materials/defects/defect_planar_grain_boundary_2d_boron_nitride_5_7/li2015-fig4b-ldos.webp" alt="Li et al. (2015), Fig. 4b: LDOS of the 4|8 and 5|7 boundaries and of perfect BN" width="300"/>

In [ ]:
from matplotlib import pyplot as plt
from mat3ra.notebooks_utils.plot import display_matplotlib_figure

offset = perfect_dos[(perfect_energies >= ENERGY_WINDOW[0]) & (perfect_energies <= ENERGY_WINDOW[1])].max()
figure, axes = plt.subplots()
axes.plot(shifted_energies, boundary_ldos + offset, color="red", label="5|7 boundary (Type III)")
axes.plot(perfect_energies, perfect_dos, color="black", label="Perfect BN")
for number, peak in enumerate(peaks, start=1):
    peak_height = offset + boundary_ldos[np.argmin(np.abs(shifted_energies - peak))]
    axes.annotate(f"({number})", (peak, peak_height), color="green")
axes.set_xlim(*ENERGY_WINDOW)
axes.set_xlabel("Energy (eV)")
axes.set_ylabel("LDOS (a.u.)")
axes.set_yticks([])
axes.legend()
display_matplotlib_figure(figure)

## 9. Comparison with Li et al. (2015)

The manuscript's values: the gap from peak (2) to peak (3), "~3.2 eV" in the text; the peaks and the perfect layer's gap
read off Fig. 4b.

In [ ]:
LI_2015 = {"gap": 3.21, "peaks": [-2.85, -1.82, 1.39, 2.31], "perfect_gap": 4.4}  # eV


def print_gap_and_peaks(gap, peaks, perfect_gap):
    print(f"Gap, peak (2) to peak (3): {gap:.2f} eV")
    for number, peak in enumerate(peaks, start=1):
        print(f"Peak ({number}): {peak:+.2f} eV")
    print(f"Perfect layer gap: {perfect_gap:.2f} eV")


print(f"Band gap and in-gap peaks of {dos_structure.name}:")
print_gap_and_peaks(gap, peaks, perfect_gap)
print("\nBand gap and in-gap peaks from the manuscript:")
print_gap_and_peaks(**LI_2015)

## References

[1] Qiucheng Li, Xiaolong Zou, Mengxi Liu, Jingyu Sun, Yabo Gao, Yue Qi, Xiebo Zhou, Boris I. Yakobson, Yanfeng Zhang, Zhongfan Liu, "Grain Boundary Structures and Electronic Properties of Hexagonal Boron Nitride on Cu(111)", Nano Lett. 15(9), 5804-5810 (2015). [DOI: 10.1021/acs.nanolett.5b01852](https://doi.org/10.1021/acs.nanolett.5b01852)

[2] K. F. Garrity, J. W. Bennett, K. M. Rabe and D. Vanderbilt, "Pseudopotentials for high-throughput DFT calculations", Comput. Mater. Sci. 81, 446 (2014). https://doi.org/10.1016/j.commatsci.2013.08.053